In [1]:
import pandas as pd 
import joblib 
from sklearn.ensemble import RandomForestClassifier
import os
import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from PIL import Image

In [4]:
raw_data = datasets.ImageFolder(root="E:\psoriasis\SKIN DISEASE CLASSIFICATION FOR SKIN ANALYSIS TOOL")
class_names = raw_data.classes


In [5]:
weight = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights= weight)

In [6]:
model.classifier = torch.nn.Identity()
model.eval()
transform = weight.transforms()

In [11]:
all_combined_features = []
all_labels = []
print("Extracting the features")
for path, Label in raw_data.samples:
    image = Image.open(path).convert("RGB")
    image_transformed = transform(image)
    image_tensor = image_transformed.unsqueeze(0)

    with torch.no_grad():
        image_features = model(image_tensor).numpy().flatten()

        
    age = np.random.randint(0,80)/100.0
    gender = np.random.choice([0,1])
    meta_features = np.array([age,gender],dtype = np.float32)

    combined_features = np.concatenate((image_features, meta_features))

    all_combined_features.append(combined_features)
    all_labels.append(Label)
print("efficientnet trained succesfully")

Extracting the features
efficientnet trained succesfully


In [18]:
print("Random forest training..")
x = np.array(all_combined_features)
y = np.array(all_labels)

x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=32,test_size=0.25, stratify=y)

rf_model = RandomForestClassifier(n_estimators=200, random_state=32)
rf_model.fit(all_combined_features,all_labels)
print("training completed")

 mRandom forest training..
training completed


In [21]:
y_train_pred = rf_model.predict(x_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

y_test_pred = rf_model.predict(x_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

# 7. Print results
print("\n" + "="*30)
print("     MODEL PERFORMANCE")
print("="*30)
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Testing Accuracy:  {test_accuracy * 100:.2f}%")
print("="*30)


     MODEL PERFORMANCE
Training Accuracy: 100.00%
Testing Accuracy:  100.00%


In [22]:
from torchinfo import summary
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

# 1. Load EfficientNet-B0 and set classifier to Identity
weights = EfficientNet_B0_Weights.DEFAULT
efficientnet = efficientnet_b0(weights=weights)
efficientnet.classifier = nn.Identity()

# 2. Print complete model summary
# input_size format: (batch_size, channels, height, width)
summary(efficientnet, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [1, 1280]                 --
├─Sequential: 1-1                                       [1, 1280, 7, 7]           --
│    └─Conv2dNormActivation: 2-1                        [1, 32, 112, 112]         --
│    │    └─Conv2d: 3-1                                 [1, 32, 112, 112]         864
│    │    └─BatchNorm2d: 3-2                            [1, 32, 112, 112]         64
│    │    └─SiLU: 3-3                                   [1, 32, 112, 112]         --
│    └─Sequential: 2-2                                  [1, 16, 112, 112]         --
│    │    └─MBConv: 3-4                                 [1, 16, 112, 112]         1,448
│    └─Sequential: 2-3                                  [1, 24, 56, 56]           --
│    │    └─MBConv: 3-5                                 [1, 24, 56, 56]           6,004
│    │    └─MBConv: 3-6                              

In [23]:
class_names

['Acne',
 'Bullous Disease Photos',
 'Cellulitis Impetigo and other Bacterial Infections',
 'Eczema Photos',
 'Exanthems and Drug Eruptions',
 'Hair Loss Photos Alopecia and other Hair Diseases',
 'Herpes HPV and other STDs Photos',
 'Light Diseases and Disorders of Pigmentation',
 'Melanoma Skin Cancer Nevi and Moles',
 'Nail Fungus and other Nail Disease',
 'Psoriasis pictures Lichen Planus and related diseases',
 'Scabies Lyme Disease and other Infestations and Bites',
 'Tinea Ringworm Candidiasis and other Fungal Infections',
 'Vascular Tumors',
 'normal']

In [28]:
img_path = r"E:\psoriasis\SKIN DISEASE CLASSIFICATION FOR SKIN ANALYSIS TOOL\Nail Fungus and other Nail Disease\acute-paronychia-23.jpg"
test_image = Image.open(img_path).convert("RGB")
test_image_tensor = transform(test_image).unsqueeze(0)

with torch.no_grad():
    test_image_feature = model(test_image_tensor).numpy()[0]

test_meta_feature = np.array([45/100.0,1], dtype = np.float32)

combined_input = np.concatenate((test_image_feature,test_meta_feature)).reshape(1,-1)

prediction_idx = rf_model.predict(combined_input)[0]
predicted_class = raw_data.classes[prediction_idx]
confidence = np.max(rf_model.predict_proba(combined_input)[0]) * 100

# 6. Display results and consultation advisory
print("=" * 50)
print("             PREDICTION RESULTS")
print("=" * 50)
print(f"Detected Condition : {predicted_class}")
print(f"Confidence Level   : {confidence:.2f}%")
print("-" * 50)

             PREDICTION RESULTS
Detected Condition : Nail Fungus and other Nail Disease
Confidence Level   : 76.50%
--------------------------------------------------


In [ ]:
import joblib

# 1. Save the trained Random Forest model
joblib.dump(rf_model, 'rf_model.pkl')

# 2. Save the list of class names
joblib.dump(raw_data.classes, 'class_names.pkl')

print("Model and class names successfully saved!")

In [30]:
import os 
import joblib
dir = os.chdir(r"D:\mini_project\new")
# 1. Save the trained Random Forest model
joblib.dump(rf_model, 'rf_model.pkl')

# 2. Save the list of class names
joblib.dump(raw_data.classes, 'class_names.pkl')

print("Model and class names successfully saved!")

Model and class names successfully saved!
